Linear Algebra 2 - Applied Linear Algebra for Deep Learning

Section 19: Applied Linear Algebra for Deep Learning
Scaled dot-product attention, multi-head attention,
weight initialisation theory (Xavier and He),
batch normalisation and layer normalisation from scratch,
gradient flow and vanishing/exploding gradient analysis.

Grand Example: mini transformer block combining all topics.

Section 19 - Scaled Dot-Product Attention

Attention maps each query to a weighted sum of values, where weights come from
how much each query-key pair aligns (dot product similarity).

Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

The 1/sqrt(d_k) scaling prevents dot products from growing too large in high dimensions,
which would push softmax into regions of near-zero gradient.

Shapes:
Q: (seq, d_k)  -- queries
K: (seq, d_k)  -- keys
V: (seq, d_v)  -- values
output: (seq, d_v)

In [ ]:
import numpy as np

# Base Python / NumPy: scaled dot-product attention from scratch

def softmax(x):
    # numerically stable softmax
    x_shift = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x_shift)
    return e / e.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k   = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (seq_q, seq_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)   # mask out future positions
    weights = softmax(scores)             # (seq_q, seq_k)
    output  = weights @ V                 # (seq_q, d_v)
    return output, weights

np.random.seed(42)
seq, d_k, d_v = 6, 8, 8
Q = np.random.randn(seq, d_k)
K = np.random.randn(seq, d_k)
V = np.random.randn(seq, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f'attention output shape : {output.shape}')
print(f'attention weights shape: {weights.shape}')
print(f'weights sum to 1 per row: {np.allclose(weights.sum(axis=1), 1.0)}')

# causal mask: position i can only attend to positions <= i
causal_mask = np.tril(np.ones((seq, seq), dtype=bool))
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f'\nweights upper triangle (should be 0):\n{weights_causal.round(4)}')

In [ ]:
import numpy as np

# Multi-head attention from scratch
# Split Q, K, V into h heads, run attention in parallel, concatenate and project

def multihead_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    seq, d_model = X.shape
    d_k = d_model // num_heads

    Q_all = X @ W_Q  # (seq, d_model)
    K_all = X @ W_K
    V_all = X @ W_V

    head_outputs = []
    for h in range(num_heads):
        Q_h = Q_all[:, h*d_k:(h+1)*d_k]  # (seq, d_k)
        K_h = K_all[:, h*d_k:(h+1)*d_k]
        V_h = V_all[:, h*d_k:(h+1)*d_k]
        scores  = Q_h @ K_h.T / np.sqrt(d_k)
        weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
        weights /= weights.sum(axis=-1, keepdims=True)
        head_outputs.append(weights @ V_h)

    concat = np.concatenate(head_outputs, axis=-1)  # (seq, d_model)
    return concat @ W_O

np.random.seed(0)
seq, d_model, num_heads = 5, 16, 4
scale = np.sqrt(2.0 / d_model)

X   = np.random.randn(seq, d_model)
W_Q = np.random.randn(d_model, d_model) * scale
W_K = np.random.randn(d_model, d_model) * scale
W_V = np.random.randn(d_model, d_model) * scale
W_O = np.random.randn(d_model, d_model) * scale

out = multihead_attention(X, W_Q, W_K, W_V, W_O, num_heads)
print(f'multi-head attention output shape: {out.shape}  (seq={seq}, d_model={d_model})')

In [ ]:
import torch
import torch.nn.functional as F

# PyTorch: attention using einsum for clarity

def attention_pytorch(Q, K, V):
    d_k   = Q.shape[-1]
    scores = torch.einsum('bqd,bkd->bqk', Q, K) / d_k**0.5
    weights = F.softmax(scores, dim=-1)
    return torch.einsum('bqk,bkd->bqd', weights, V), weights

batch, seq, d_k = 2, 6, 8
Q = torch.randn(batch, seq, d_k)
K = torch.randn(batch, seq, d_k)
V = torch.randn(batch, seq, d_k)

out, w = attention_pytorch(Q, K, V)
print(f'output shape : {out.shape}')
print(f'weights shape: {w.shape}')
print(f'weights sum to 1: {torch.allclose(w.sum(dim=-1), torch.ones(batch, seq))}')

# compare with PyTorch built-in
out_ref = F.scaled_dot_product_attention(Q, K, V)
print(f'matches F.scaled_dot_product_attention: {torch.allclose(out, out_ref, atol=1e-5)}')

In [ ]:
import tensorflow as tf

# TensorFlow: attention using tf.einsum

batch, seq, d_k = 2, 6, 8
Q = tf.random.normal([batch, seq, d_k])
K = tf.random.normal([batch, seq, d_k])
V = tf.random.normal([batch, seq, d_k])

scores  = tf.einsum('bqd,bkd->bqk', Q, K) / tf.cast(d_k, tf.float32)**0.5
weights = tf.nn.softmax(scores, axis=-1)
output  = tf.einsum('bqk,bkd->bqd', weights, V)

print(f'TF attention output shape : {output.shape}')
print(f'TF attention weights shape: {weights.shape}')

Weight Initialisation Theory (Xavier and He)

Goal: keep variance of activations roughly constant across layers.
If weights are too large, activations explode. Too small, they vanish.

Xavier / Glorot (for tanh and sigmoid):
Var(W) = 2 / (fan_in + fan_out)
Ensures variance of output equals variance of input (in expectation).

He / Kaiming (for ReLU):
Var(W) = 2 / fan_in
Corrects for the fact that ReLU kills roughly half the neurons.

Both derive from requiring that the variance of z = W x satisfies Var(z) = Var(x).

In [ ]:
import numpy as np

def xavier_init(fan_in, fan_out):
    std = np.sqrt(2.0 / (fan_in + fan_out))
    return np.random.randn(fan_in, fan_out) * std

def he_init(fan_in, fan_out):
    std = np.sqrt(2.0 / fan_in)
    return np.random.randn(fan_in, fan_out) * std

np.random.seed(0)
layer_sizes = [256, 128, 64, 32, 16]

print('Variance propagation through 4 linear layers (tanh, Xavier init):')
x = np.random.randn(1000, layer_sizes[0])  # batch of 1000
print(f'  input var: {x.var():.4f}')
for i in range(len(layer_sizes) - 1):
    W = xavier_init(layer_sizes[i], layer_sizes[i+1])
    x = np.tanh(x @ W)
    print(f'  layer {i+1} output var: {x.var():.4f}')

print('\nVariance propagation (ReLU, He init):')
x = np.random.randn(1000, layer_sizes[0])
print(f'  input var: {x.var():.4f}')
for i in range(len(layer_sizes) - 1):
    W = he_init(layer_sizes[i], layer_sizes[i+1])
    x = np.maximum(0, x @ W)   # ReLU
    print(f'  layer {i+1} output var: {x.var():.4f}')

print('\nVariance propagation (ReLU, naive std=0.01 -- vanishes):')
x = np.random.randn(1000, layer_sizes[0])
print(f'  input var: {x.var():.6f}')
for i in range(len(layer_sizes) - 1):
    W = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01
    x = np.maximum(0, x @ W)
    print(f'  layer {i+1} output var: {x.var():.2e}')

Batch Normalisation and Layer Normalisation

Batch Normalisation (BN): normalises each feature across the batch dimension.
For a batch X (batch, features): mu = mean over batch, sigma = std over batch.
x_norm = (x - mu) / (sigma + eps), then x_out = gamma * x_norm + beta.
At inference, uses running statistics accumulated during training.

Layer Normalisation (LN): normalises each sample across its feature dimension.
For a single sample x (features,): normalise over all features for that sample.
LN is used in transformers because it is independent of batch size.

In [ ]:
import numpy as np

class BatchNorm:
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.gamma   = np.ones(num_features)   # learnable scale
        self.beta    = np.zeros(num_features)  # learnable shift
        self.eps     = eps
        self.momentum = momentum
        self.running_mean = np.zeros(num_features)
        self.running_var  = np.ones(num_features)

    def forward(self, X, training=True):
        if training:
            mu    = X.mean(axis=0)                   # per feature mean over batch
            sigma = X.std(axis=0)                    # per feature std
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mu
            self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * sigma**2
        else:
            mu    = self.running_mean
            sigma = np.sqrt(self.running_var)
        X_norm = (X - mu) / (sigma + self.eps)
        return self.gamma * X_norm + self.beta

np.random.seed(5)
X = np.random.randn(32, 8) * 3 + 2   # batch=32, features=8, non-zero mean
bn = BatchNorm(num_features=8)
X_out = bn.forward(X, training=True)

print(f'input  mean: {X.mean(axis=0).round(4)}')
print(f'output mean: {X_out.mean(axis=0).round(6)}  (should be ~0)')
print(f'output std : {X_out.std(axis=0).round(4)}   (should be ~1)')

# Layer Normalisation
def layer_norm(x, gamma, beta, eps=1e-5):
    # x: (batch, features) -- normalise over feature dim for each sample
    mu    = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1,  keepdims=True)
    return gamma * (x - mu) / (sigma + eps) + beta

gamma = np.ones(8)
beta  = np.zeros(8)
X_ln  = layer_norm(X, gamma, beta)

print(f'\nLayer Norm per-sample mean: {X_ln.mean(axis=1)[:5].round(6)}  (should be ~0)')
print(f'Layer Norm per-sample std : {X_ln.std(axis=1)[:5].round(4)}  (should be ~1)')

In [ ]:
import numpy as np

# Gradient flow: how gradient magnitude changes through layers
# dL/dx^(l) = dL/dx^(l+1) @ W^(l+1).T   (backprop through a linear layer)
# gradient norms grow/shrink based on singular values of W

np.random.seed(0)
num_layers = 10
d = 64

grad_norms_naive  = [1.0]
grad_norms_xavier = [1.0]

for _ in range(num_layers):
    W_naive  = np.random.randn(d, d) * 1.0
    W_xavier = np.random.randn(d, d) * np.sqrt(1.0 / d)

    g = np.ones(d)

    # backprop through one layer: g = W.T @ g (simplified, no activation)
    g_naive  = W_naive.T  @ (np.ones(d) * grad_norms_naive[-1]  / d)
    g_xavier = W_xavier.T @ (np.ones(d) * grad_norms_xavier[-1] / d)

    grad_norms_naive.append(np.linalg.norm(g_naive))
    grad_norms_xavier.append(np.linalg.norm(g_xavier))

print('gradient norm through 10 layers:')
print(f'naive (std=1.0)  : {[f"{v:.2e}" for v in grad_norms_naive]}')
print(f'xavier (1/sqrt(d)): {[f"{v:.3f}" for v in grad_norms_xavier]}')

Grand Example - Mini Transformer Block

A transformer block combines all the topics in this notebook:
1. Multi-head self-attention (scaled dot product, einsum)
2. Layer normalisation (normalise across features per token)
3. Feed-forward network (two linear layers with ReLU, He init)
4. Residual connections (add input to output, prevents vanishing gradient)

Forward pass: output = LayerNorm(x + FFN(LayerNorm(x + MultiHeadAttention(x))))

In [ ]:
import numpy as np

np.random.seed(1)

def softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def layer_norm_np(x, eps=1e-5):
    mu  = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1,  keepdims=True)
    return (x - mu) / (std + eps)

def mha(X, WQ, WK, WV, WO, h):
    seq, d = X.shape
    dk = d // h
    Q, K, V = X @ WQ, X @ WK, X @ WV
    heads = []
    for i in range(h):
        Qi = Q[:, i*dk:(i+1)*dk]
        Ki = K[:, i*dk:(i+1)*dk]
        Vi = V[:, i*dk:(i+1)*dk]
        w = softmax_np(Qi @ Ki.T / dk**0.5)
        heads.append(w @ Vi)
    return np.concatenate(heads, axis=-1) @ WO

def ffn(X, W1, b1, W2, b2):
    return np.maximum(0, X @ W1 + b1) @ W2 + b2   # ReLU activation

# hyperparameters
seq, d_model, h, d_ff = 8, 32, 4, 64
s = np.sqrt(2.0 / d_model)

# Xavier/He initialised weights
WQ = np.random.randn(d_model, d_model) * s
WK = np.random.randn(d_model, d_model) * s
WV = np.random.randn(d_model, d_model) * s
WO = np.random.randn(d_model, d_model) * s
W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)  # He init for ReLU
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * s
b2 = np.zeros(d_model)

# input embeddings
X = np.random.randn(seq, d_model)

# --- transformer block forward pass ---
# 1. multi-head self-attention + residual + layer norm
attn_out = mha(X, WQ, WK, WV, WO, h)
X2 = layer_norm_np(X + attn_out)           # residual connection

# 2. feed-forward + residual + layer norm
ff_out = ffn(X2, W1, b1, W2, b2)
X3 = layer_norm_np(X2 + ff_out)            # residual connection

print(f'input shape  : {X.shape}')
print(f'attn output  : {attn_out.shape}')
print(f'after LN 1   : {X2.shape}')
print(f'ffn output   : {ff_out.shape}')
print(f'final output : {X3.shape}')

print(f'\noutput stats:')
print(f'mean per token: {X3.mean(axis=-1).round(4)}  (LN ensures ~0)')
print(f'std  per token: {X3.std(axis=-1).round(4)}   (LN ensures ~1)')